In [13]:
import pandas as pd
import networkx as nx
from collections import defaultdict
import numpy as np

In [ ]:
df_2015 = pd.read_csv("../Data/events_2015_full.csv")
df_2025 = pd.read_csv("../Data/events_2025_full.csv")

# Convert timestamps
df_2015["entry_time"] = pd.to_datetime(df_2015["entry_time"])
df_2015["exit_time"] = pd.to_datetime(df_2015["exit_time"])
df_2025["entry_time"] = pd.to_datetime(df_2025["entry_time"])
df_2025["exit_time"] = pd.to_datetime(df_2025["exit_time"])

# Sort globally (important)
df_2015 = df_2015.sort_values(["ship_id", "entry_time"])
df_2025 = df_2025.sort_values(["ship_id", "entry_time"])

MIN_TIME_GAP_HRS = 1

df_2015["gap"] = df_2015.groupby("ship_id")["entry_time"].diff().dt.total_seconds() / 3600
df_2025["gap"] = df_2025.groupby("ship_id")["entry_time"].diff().dt.total_seconds() / 3600

df_2015 = df_2015[(df_2015["gap"].isna()) | (df_2015["gap"] < 1000)]  # tune threshold
df_2025 = df_2025[(df_2025["gap"].isna()) | (df_2025["gap"] < 1000)]  # tune threshold

In [3]:
def validate_ports(df):
    print("Total rows:", len(df))
    
    missing_id = df["port_id"].isna().sum()
    missing_name = df["port_name"].isna().sum()
    
    print("Missing port_id:", missing_id)
    print("Missing port_name:", missing_name)
    
    unique_ports = df["port_id"].nunique()
    print("Unique ports:", unique_ports)

validate_ports(df_2015)
validate_ports(df_2025)

Total rows: 2211877
Missing port_id: 0
Missing port_name: 546828
Unique ports: 7138
Total rows: 2448119
Missing port_id: 0
Missing port_name: 639493
Unique ports: 7788


In [4]:
def build_ship_sequences(df):
    ship_sequences = {}

    for ship_id, group in df.groupby("ship_id"):
        group = group.sort_values("entry_time")
        
        ports = group["port_id"].tolist()
        
        # Remove consecutive duplicates (important!)
        cleaned_ports = []
        for p in ports:
            if len(cleaned_ports) == 0 or cleaned_ports[-1] != p:
                cleaned_ports.append(p)
        
        if len(cleaned_ports) > 1:
            ship_sequences[ship_id] = cleaned_ports

    return ship_sequences


ship_sequences_2015 = build_ship_sequences(df_2015)
ship_sequences_2025 = build_ship_sequences(df_2025)

print("Total ships:", len(ship_sequences_2015))
print("Total ships:", len(ship_sequences_2025))

Total ships: 31936
Total ships: 38689


In [5]:
def build_graph(ship_sequences):
    edge_weights = defaultdict(int)

    for ship_id, ports in ship_sequences.items():
        for i in range(len(ports) - 1):
            A = ports[i]
            B = ports[i+1]

            if A != B:
                edge_weights[(A, B)] += 1

    return edge_weights

edge_weights_2015 = build_graph(ship_sequences_2015)
edge_weights_2025 = build_graph(ship_sequences_2025)

In [6]:
def create_networkx_graph(edge_weights):
    G = nx.DiGraph()

    for (A, B), w in edge_weights.items():
        G.add_edge(A, B, weight=w)

    return G


G_2015 = create_networkx_graph(edge_weights_2015)
G_2025 = create_networkx_graph(edge_weights_2025)

print("Nodes in 2015 graph:", G_2015.number_of_nodes())
print("Edges in 2015 graph:", G_2015.number_of_edges())
print("Nodes in 2025 graph:", G_2025.number_of_nodes())
print("Edges in 2025 graph:", G_2025.number_of_edges())

Nodes in 2015 graph: 7097
Edges in 2015 graph: 188545
Nodes in 2025 graph: 7717
Edges in 2025 graph: 182300


In [7]:
def add_port_metadata(G, df):
    port_info = df.drop_duplicates("port_id")[["port_id", "port_name", "lat", "lon"]]

    for _, row in port_info.iterrows():
        port_id = row["port_id"]
        
        if port_id in G:
            G.nodes[port_id]["name"] = row["port_name"]
            G.nodes[port_id]["lat"] = row["lat"]
            G.nodes[port_id]["lon"] = row["lon"]


add_port_metadata(G_2015, df_2015)
add_port_metadata(G_2025, df_2025)

In [10]:
edges_2015 = []
edges_2025 = []

for u, v, data in G_2015.edges(data=True):
    edges_2015.append({
        "source": u,
        "target": v,
        "weight": data["weight"]
    })

for u, v, data in G_2025.edges(data=True):
    edges_2025.append({
        "source": u,
        "target": v,
        "weight": data["weight"]
    })

edges_df_2015 = pd.DataFrame(edges_2015)
edges_df_2015.to_csv("../EdgeList/shipping_network_2015.csv", index=False)

edges_df_2025 = pd.DataFrame(edges_2025)
edges_df_2025.to_csv("../EdgeList/shipping_network_2025.csv", index=False)

In [17]:
# Top busiest routes
edges_df_2015.sort_values("weight", ascending=False).head(10)
edges_df_2025.sort_values("weight", ascending=False).head(10)

# Degree
print("Avg degree (2015): ", sum(dict(G_2015.degree()).values()) / G_2015.number_of_nodes())
print("Avg in-degree (2015):", sum(dict(G_2015.in_degree()).values()) / G_2015.number_of_nodes())
print("Avg out-degree (2015):", sum(dict(G_2015.out_degree()).values()) / G_2015.number_of_nodes())
print("Avg clustering coefficient (2015):", nx.average_clustering(G_2015.to_undirected()))
# print("Avg charateristic path length (2015):", np.mean([nx.average_shortest_path_length(G_2015.to_undirected().subgraph(comp)) for comp in nx.connected_components(G_2015.to_undirected())]))

print("Avg degree (2025): ", sum(dict(G_2025.degree()).values()) / G_2025.number_of_nodes())
print("Avg in-degree (2025):", sum(dict(G_2025.in_degree()).values()) / G_2025.number_of_nodes())
print("Avg out-degree (2025):", sum(dict(G_2025.out_degree()).values()) / G_2025.number_of_nodes())
print("Avg clustering coefficient (2025):", nx.average_clustering(G_2025.to_undirected()))
# print("Avg charateristic path length (2025):", np.mean([nx.average_shortest_path_length(G_2025.to_undirected().subgraph(comp)) for comp in nx.connected_components(G_2025.to_undirected())]))

Avg degree (2015):  53.133718472594055
Avg in-degree (2015): 26.566859236297027
Avg out-degree (2015): 26.566859236297027
Avg clustering coefficient (2015): 0.5222967035575492
Avg degree (2025):  47.24633925100428
Avg in-degree (2025): 23.62316962550214
Avg out-degree (2025): 23.62316962550214
Avg clustering coefficient (2025): 0.4884025913458535
